In [40]:
import yfinance as yf
import pandas as pd
import numpy as np

In [41]:
ticker_symbol = "AAPL"


apple_stock = yf.Ticker(ticker_symbol)


historical_data = apple_stock.history(period="10y")


print(historical_data)

                                 Open        High         Low       Close  \
Date                                                                        
2016-11-07 00:00:00-05:00   25.302968   25.401808   25.160454   25.378822   
2016-11-08 00:00:00-05:00   25.355828   25.679931   25.215614   25.528223   
2016-11-09 00:00:00-05:00   25.256987   25.587986   24.836345   25.486847   
2016-11-10 00:00:00-05:00   25.535123   25.535123   24.326062   24.776587   
2016-11-11 00:00:00-05:00   24.622581   25.024835   24.491561   24.923697   
...                               ...         ...         ...         ...   
2026-11-02 00:00:00-05:00  342.670013  343.799988  335.820007  338.320007   
2026-11-03 00:00:00-05:00  325.579987  334.500000  325.329987  331.899994   
2026-11-04 00:00:00-05:00  341.570007  354.500000  340.989990  349.510010   
2026-11-05 00:00:00-05:00  353.890015  355.380005  342.059998  344.170013   
2026-11-06 00:00:00-05:00  338.730011  345.790009  338.299988  342.369995   

In [42]:
historical_data['Date'] = historical_data.index

historical_data['Date']  = pd.to_datetime(historical_data['Date']).dt.date
historical_data.index = pd.to_datetime(historical_data['Date']).dt.date


In [43]:
historical_data = historical_data.drop(columns=['Dividends', 'Stock Splits', 'Date'])

In [44]:

rng = np.random.default_rng(10)         
rules = []

for i in range(500):
    kind = rng.choice(['ma', 'rsi', 'bb'])
    if kind == 'ma':
        fast = int(rng.integers(5, 60))
        rules.append({'kind': 'ma', 'fast': fast, 'slow': int(rng.integers(fast + 10, 250))})
    elif kind == 'rsi':
        rules.append({'kind': 'rsi', 'n': int(rng.integers(5, 30)),
                      'buy': int(rng.integers(15, 40)), 'exit': int(rng.integers(45, 75))})
    else:
        rules.append({'kind': 'bb', 'n': int(rng.integers(10, 60)),
                      'k': round(float(rng.uniform(1.0, 3.0)), 2),
                      'mode': str(rng.choice(['reversion', 'breakout']))})

rules = pd.DataFrame(rules)

In [47]:

close = historical_data['Close']
COST = 0.0005  


def rsi(p, n):
    d = p.diff()
    return 100 - 100 / (1 + d.clip(lower=0).rolling(n).mean() / (-d).clip(lower=0).rolling(n).mean())

def sharpe(r):
    sd = r.std()
    s = r.mean() / sd * np.sqrt(252)
    return s.where(sd > 0, 0.0) if isinstance(s, pd.Series) else (s if sd > 0 else 0.0)

def position(rule, p):
    if rule['kind'] == 'ma':
        slow = p.rolling(int(rule['slow'])).mean()
        pos = (p.rolling(int(rule['fast'])).mean() > slow).astype(float)
        return pos.where(slow.notna(), 0.0)
    sig = pd.Series(np.nan, index=p.index)
    if rule['kind'] == 'rsi':
        r = rsi(p, int(rule['n']))
        sig[r < rule['buy']] = 1.0
        sig[r > rule['exit']] = 0.0
    else:
        mid, sd = p.rolling(int(rule['n'])).mean(), p.rolling(int(rule['n'])).std()
        lower, upper = mid - rule['k'] * sd, mid + rule['k'] * sd
        if rule['mode'] == 'reversion':
            sig[p < lower] = 1.0; sig[p > upper] = 0.0
        else:
            sig[p > upper] = 1.0; sig[p < lower] = 0.0
    return sig.ffill().fillna(0.0)

def search(p):
    pos = pd.concat({i: position(row, p) for i, row in rules.iterrows()}, axis=1)
    r = p.pct_change().fillna(0.0)
    held = pos.shift(1).fillna(0.0)
    net = held.mul(r, axis=0) - held.diff().abs().fillna(held.abs()) * COST
    split = p.index[len(p) // 2]
    is_sh = sharpe(net[net.index < split])
    best = is_sh.idxmax()
    return {'best_rule': rules.loc[best, ['kind','fast','slow','n','buy','exit','k','mode']].dropna().to_dict(),
            'best_is_sharpe': is_sh[best],
            'median_is_sharpe': is_sh.median(),
            'out_of_sample_sharpe': sharpe(net[net.index >= split][best]),
            'share_is_sharpe_above_0.5': (is_sh > 0.5).mean()}


     

In [48]:
sigma = close.pct_change().std()
rw = pd.Series(close.iloc[0] * np.exp(np.cumsum(rng.normal(-0.5 * sigma**2, sigma, len(close)))),
               index=close.index)

print('Real data:  ', search(close))
print('Random walk:', search(rw))

Real data:   {'best_rule': {'kind': 'bb', 'n': 11.0, 'k': 1.19, 'mode': 'breakout'}, 'best_is_sharpe': np.float64(1.9021177321078728), 'median_is_sharpe': np.float64(0.7665780855217366), 'out_of_sample_sharpe': np.float64(0.793301507172644), 'share_is_sharpe_above_0.5': np.float64(0.68)}
Random walk: {'best_rule': {'kind': 'rsi', 'n': 21.0, 'buy': 15.0, 'exit': 49.0}, 'best_is_sharpe': np.float64(0.27501056036543287), 'median_is_sharpe': np.float64(-0.3699249497123662), 'out_of_sample_sharpe': np.float64(0.18317014660063544), 'share_is_sharpe_above_0.5': np.float64(0.0)}
